In [ ]:
!pip install openai --quiet


In [ ]:
#Currently this is not INTERACTING with anything outside of the promts given here, that is something that needs to be worked on
import os
import csv
from datetime import datetime
from openai import OpenAI
#API key
os.environ["OPENAI_API_KEY"] = "[REDACTED_KEY]"  # <-- Replace this
client = OpenAI()


scenarios = [ #Making multiple promts
    "Book a table for 2 at an Italian restaurant in New York on Friday at 8 PM.",
    "Reserve a vegan restaurant for 6 people in Seattle next Wednesday at 6:45 PM.",
    "Find a sushi place in Los Angeles for 3 people tomorrow night at 7 PM.",
    "Book a Japanese restaurant in San Francisco for 4 people this Saturday at 7:30 PM.",
]

#Scoring, DEF NEEDS TO BE WORKED ON LATER
def score_response(text, scenario):
    score = 0
    if "2" in text or "two" in text: score += 1
    if "4" in text or "four" in text: score += 1
    if "6" in text or "six" in text: score += 1
    if "Friday" in text: score += 1
    if "Wednesday" in text: score += 1
    if "Saturday" in text: score += 1
    if "8" in text or "8 PM" in text: score += 1
    if "6:45" in text: score += 1
    if "7:30" in text: score += 1
    if "restaurant" in text.lower(): score += 1
    return min(score, 5)  # Cap at 5


results = [] #Run Benchmark
for i, scenario in enumerate(scenarios, 1):
    print(f"\n--- Scenario {i} ---")
    response = client.chat.completions.create(
        model="gpt-4",
        messages=[
            {"role": "system", "content": "You are a helpful AI agent that handles restaurant reservations."},
            {"role": "user", "content": scenario}
        ],
        temperature=0.3
    )
    result_text = response.choices[0].message.content
    score = score_response(result_text, scenario)
    print(result_text)
    print(f"Score: {score}/5")

    results.append({
        "Scenario": scenario,
        "Response": result_text,
        "Score": score,
        "Timestamp": datetime.now().isoformat()
    })

#Save results in a CSV to view results better
csv_filename = "chatgpt_reservation_benchmark.csv"
with open(csv_filename, mode='w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=results[0].keys())
    writer.writeheader()
    writer.writerows(results)

print(f"\n✅ Benchmark complete. Results saved to {csv_filename}")

In [ ]:
from google.colab import files
files.download("chatgpt_reservation_benchmark.csv")